In [43]:
# Dynamic SAmpling Policy Optimization
# 1) Dynamic Sampling of Completions (Filter Gate)
# 2)  Clip-Higher i.e lower clip and upper clip can be different
# 3) Token Level Policy Gradient loss 

import copy
import numpy as np
import torch
import torch.nn.functional as F

from transformers import AutoTokenizer, AutoModelForCausalLM

torch.manual_seed(0)
np.random.seed(0)

device = "cuda" if torch.cuda.is_available() else "cpu"
model_name = "sshleifer/tiny-gpt2"

tokenizer = AutoTokenizer.from_pretrained(model_name)

In [44]:
# ------------------------------------------------------------
# 0) Load policies
# ------------------------------------------------------------

base_policy = AutoModelForCausalLM.from_pretrained(model_name).to(device)

# Current trainable policy p
policy_new = copy.deepcopy(base_policy).to(device)

# For demo clarity, keep dropout off.
# Gradients still flow because requires_grad=True.
policy_new.eval()
for p in policy_new.parameters():
    p.requires_grad_(True)

# Frozen reference policy pi_ref
policy_reference = copy.deepcopy(base_policy).to(device).eval()
for p in policy_reference.parameters():
    p.requires_grad_(False)

print("Loaded:", model_name, "on", device)

Loaded: sshleifer/tiny-gpt2 on cpu


In [45]:
def encode_prompt_and_response(tokenizer, prompt_text, completion_text):
    """
    Returns:
      input_ids: [1, seq_len] for prompt+completion (teacher-forced)
      prompt_len: #tokens in prompt
      comp_len:   #tokens in completion
    """
    prompt_ids = tokenizer(prompt_text, return_tensors="pt", add_special_tokens=False)["input_ids"]
    full_ids   = tokenizer(prompt_text + completion_text, return_tensors="pt", add_special_tokens=False)["input_ids"]
    prompt_len = prompt_ids.shape[1]
    comp_len   = full_ids.shape[1] - prompt_len
    assert comp_len > 0, "Completion must add at least one token."
    return full_ids, prompt_len, comp_len

In [46]:
def token_logprobs_for_response(model, input_ids: torch.Tensor, prompt_len: int):
    """
    Computes per-token log-probs for response tokens.

    Causal LM convention:
        logits[:, k, :] predicts token at position k + 1

    For response token at absolute position pos:
        probability comes from logits at pos - 1.

    Returns:
        resp_token_ids: [T]
        resp_logps:     [T]
    """
    model_device = next(model.parameters()).device
    input_ids = input_ids.to(model_device)

    out = model(input_ids, use_cache=False)

    logits = out.logits  # [1, seq_len, vocab]
    logp_all = F.log_softmax(logits, dim=-1)

    token_ids = input_ids[0]  # [seq_len]
    seq_len = token_ids.shape[0]

    resp_token_ids = token_ids[prompt_len:]  # [T]

    prev_pos = torch.arange(
        prompt_len - 1,
        seq_len - 1,
        device=model_device,
    )  # [T]

    lp_rows = logp_all[0, prev_pos, :]  # [T, vocab]

    resp_logps = lp_rows.gather(
        dim=1,
        index=resp_token_ids.unsqueeze(1),
    ).squeeze(1)  # [T]

    return resp_token_ids, resp_logps

In [47]:
def per_token_logps(model, tokenizer, prompt: str, completion: str):
    """
    Teacher-force prompt + completion and score completion tokens.
    """
    input_ids, prompt_len, _ = encode_prompt_and_response(
        tokenizer,
        prompt,
        completion,
    )

    resp_ids, resp_logps = token_logprobs_for_response(
        model,
        input_ids,
        prompt_len,
    )

    return resp_ids, resp_logps

In [48]:
# ------------------------------------------------------------
# 3) Toy reward + dynamic sampling gate
# ------------------------------------------------------------

def toy_is_correct(prompt: str, completion: str) -> bool:
    """
    Toy binary correctness function.

    Real DAPO usually uses a verifier / reward model.
    """
    c = completion.lower()

    if "2 + 2" in prompt:
        return (" 4" in completion) or ("four" in c)

    if "cat is on the" in prompt:
        if "mat" in c or "runs" in c:
            return False
        return ("wall" in c) or ("floor" in c) or ("sleep" in c)

    return False

In [49]:
def toy_reward(prompt: str, completion: str) -> float:
    """
    Binary reward:
        1.0 for correct
        0.0 for incorrect
    """
    return 1.0 if toy_is_correct(prompt, completion) else 0.0

In [50]:
def dynamic_sampling_ok_binary(rewards: np.ndarray) -> bool:
    """
    DAPO-style dynamic sampling gate for binary rewards.

    Keep only groups with mixed outcomes:
        not all wrong
        not all correct

    This avoids zero-learning-signal groups.
    """
    G = int(rewards.shape[0])
    num_correct = int(np.sum(rewards > 0.5))

    return 0 < num_correct < G

In [51]:
def dynamic_sampling_ok_continuous(
    rewards: np.ndarray,
    min_std: float = 1e-6,
) -> bool:
    """
    Keep groups with meaningful reward diversity.

    Accept:
        [0.2, 0.8, 0.6]

    Reject:
        [0.7, 0.7, 0.7]
    """
    return float(np.std(rewards)) > min_std

In [52]:
def dynamic_sampling_ok(
    rewards: np.ndarray,
    reward_type: str = "binary",
    min_std: float = 1e-6,
) -> bool:
    """
    Unified dynamic sampling gate.
    """
    if reward_type == "binary":
        return dynamic_sampling_ok_binary(rewards)

    if reward_type == "continuous":
        return dynamic_sampling_ok_continuous(rewards, min_std=min_std)

    raise ValueError("reward_type must be 'binary' or 'continuous'")

In [53]:
# ------------------------------------------------------------
# 4) DAPO math helpers
# ------------------------------------------------------------

def group_relative_advantages(rewards: np.ndarray, eps: float = 1e-8):
    """
    Group-relative advantage:

        A_i = (r_i - mean(group_rewards)) / (std(group_rewards) + eps)

    One scalar advantage per completion.
    """
    mean_r = float(np.mean(rewards))
    std_r = float(np.std(rewards))

    advantages = (rewards - mean_r) / (std_r + eps)

    return advantages.astype(np.float64), mean_r, std_r

In [54]:
def dapo_ratio(logp_new: torch.Tensor, logp_old: torch.Tensor):
    """
    Tokenwise importance ratio:

        rho_t = exp(logp_new_t - logp_old_t)
              = pi_new(token_t | context) / pi_old(token_t | context)
    """
    return torch.exp(logp_new - logp_old)

In [55]:
def decoupled_clip(rho: torch.Tensor, eps_low: float, eps_high: float):
    """
    DAPO decoupled clipping / Clip-Higher idea:

        clip lower bound = 1 - eps_low
        clip upper bound = 1 + eps_high

    eps_high can be larger than eps_low. # very important
    """
    return torch.clamp(
        rho,
        min=1.0 - eps_low,
        max=1.0 + eps_high, # eps_high can be larger than eps_low very important
    )

In [56]:
def dapo_clipped_surrogate(
    rho: torch.Tensor,
    adv_tokens: torch.Tensor,
    eps_low: float,
    eps_high: float,
):
    """
    DAPO clipped surrogate:

        min(rho * A, clip(rho, 1-eps_low, 1+eps_high) * A)
    """
    rho_clip = decoupled_clip(rho, eps_low, eps_high)

    surr1 = rho * adv_tokens
    surr2 = rho_clip * adv_tokens

    surrogate = torch.minimum(surr1, surr2)

    return surrogate, rho_clip, surr1, surr2

In [57]:
def kl_estimator_deepseekmath(logp_new: torch.Tensor, logp_ref: torch.Tensor):
    """
    Positive KL estimator used in GRPO/DAPO-style objectives:

        x = logp_ref - logp_new
        ratio = exp(x) = pi_ref / pi_new
        KL_est = ratio - x - 1

    Returns:
        kl_per_token: [T]
    """
    x = logp_ref - logp_new
    ratio = torch.exp(x)

    return ratio - x - 1.0

In [58]:
def dapo_token_level_reduce(per_completion_terms: list[torch.Tensor]):
    """
    DAPO token-level reduction:

        J = sum_i sum_t term_{i,t} / sum_i T_i

    This differs from sample-level GRPO reduction:
        mean_i mean_t term_{i,t}
    """
    total_sum = torch.stack([terms.sum() for terms in per_completion_terms]).sum()
    total_tokens = int(sum(terms.numel() for terms in per_completion_terms))

    return total_sum / float(total_tokens)

In [59]:
# ------------------------------------------------------------
# 5) Collect rollout group
# ------------------------------------------------------------

@torch.no_grad()
def collect_dapo_rollout_group(
    policy_old,
    policy_reference,
    tokenizer,
    prompt: str,
    completions: list[str],
    rewards: np.ndarray,
):
    """
    Rollout-time DAPO quantities.

    For one prompt group:
        - detached old log-probs from policy_old
        - detached reference log-probs from policy_reference
        - group-relative advantage per completion
    """
    advantages, mean_r, std_r = group_relative_advantages(rewards)

    rollout_items = []

    for i, completion in enumerate(completions):
        ids_old, logp_old = per_token_logps(
            policy_old,
            tokenizer,
            prompt,
            completion,
        )

        ids_ref, logp_ref = per_token_logps(
            policy_reference,
            tokenizer,
            prompt,
            completion,
        )

        assert torch.equal(ids_old, ids_ref), (
            "Token mismatch between old policy and reference scoring."
        )

        rollout_items.append(
            {
                "completion": completion,
                "token_ids": ids_old.detach().cpu(),
                "logp_old": logp_old.detach().cpu(),
                "logp_ref": logp_ref.detach().cpu(),
                "reward": float(rewards[i]),
                "advantage": float(advantages[i]),
            }
        )

        group_stats = {
        "mean_reward": mean_r,
        "std_reward": std_r,
        "advantages": advantages,
    }

    return rollout_items, group_stats


In [60]:
# ------------------------------------------------------------
# 6) DAPO objective computation
# ------------------------------------------------------------

def dapo_compute_objective(
    policy_new,
    tokenizer,
    prompt: str,
    rollout_items: list[dict],
    beta_kl: float,
    eps_low: float,
    eps_high: float,
):
    """
    Computes DAPO objective over one prompt group.

    For each completion i:
        A_i = group-relative scalar advantage
        A_i is broadcast to all tokens in that completion

        rho_t = exp(logp_new_t - logp_old_t)

        surrogate_t = min(
            rho_t * A_i,
            clip(rho_t, 1-eps_low, 1+eps_high) * A_i
        )

        KL_t = positive KL estimator between policy_new and policy_reference

        term_t = surrogate_t - beta_kl * KL_t

    DAPO token-level reduction:

        J = sum_i sum_t term_{i,t} / sum_i T_i

    Loss:

        loss = -J
    """
    per_completion_terms = []

    all_rhos = []
    all_kls = []

    for item in rollout_items:
        completion = item["completion"]

        token_ids_old = item["token_ids"].to(device)
        logp_old = item["logp_old"].to(device)
        logp_ref = item["logp_ref"].to(device)

        adv_scalar = item["advantage"]

        ids_new, logp_new = per_token_logps(
            policy_new,
            tokenizer,
            prompt,
            completion,
        )

        assert torch.equal(ids_new, token_ids_old), (
            "Token mismatch between rollout tokens and current-policy scoring."
        )

        T = logp_new.shape[0]

        adv_tokens = torch.full(
            size=(T,),
            fill_value=adv_scalar,
            device=device,
            dtype=logp_new.dtype,
        )

        rho = dapo_ratio(logp_new, logp_old)

        surrogate, rho_clip, surr1, surr2 = dapo_clipped_surrogate(
            rho,
            adv_tokens,
            eps_low=eps_low,
            eps_high=eps_high,
        )

        kl = kl_estimator_deepseekmath(
            logp_new,
            logp_ref,
        )

        token_terms = surrogate - beta_kl * kl

        per_completion_terms.append(token_terms)

        all_rhos.append(rho)
        all_kls.append(kl)

    J = dapo_token_level_reduce(per_completion_terms)
    loss = -J

    rho_cat = torch.cat([r.detach() for r in all_rhos])
    kl_cat = torch.cat([k.detach() for k in all_kls])

    with torch.no_grad():
        clipfrac = (
            ((rho_cat > 1.0 + eps_high) | (rho_cat < 1.0 - eps_low))
            .float()
            .mean()
            .item()
        )

        stats = {
            "loss": float(loss.detach().cpu()),
            "objective": float(J.detach().cpu()),
            "mean_rho": float(rho_cat.mean().cpu()),
            "min_rho": float(rho_cat.min().cpu()),
            "max_rho": float(rho_cat.max().cpu()),
            "clipfrac": clipfrac,
            "mean_kl": float(kl_cat.mean().cpu()),
        }

    return loss, stats

In [61]:
# ------------------------------------------------------------
# 7) DAPO update step
# ------------------------------------------------------------

def dapo_update_step(
    policy_new,
    optimizer,
    tokenizer,
    prompt: str,
    rollout_items: list[dict],
    beta_kl: float,
    eps_low: float,
    eps_high: float,
):
    """
    One DAPO optimization step over one prompt group.
    """
    loss, stats = dapo_compute_objective(
        policy_new,
        tokenizer,
        prompt,
        rollout_items,
        beta_kl=beta_kl,
        eps_low=eps_low,
        eps_high=eps_high,
    )

    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

    return stats


In [62]:
# ------------------------------------------------------------
# 8) Evaluation helper
# ------------------------------------------------------------

@torch.no_grad()
def dapo_eval_stats(
    policy_new,
    tokenizer,
    prompt: str,
    rollout_items: list[dict],
    beta_kl: float,
    eps_low: float,
    eps_high: float,
):
    """
    Compute DAPO stats without updating.

    Note:
        This uses no_grad, so it is for logging only.
    """
    loss, stats = dapo_compute_objective(
        policy_new,
        tokenizer,
        prompt,
        rollout_items,
        beta_kl=beta_kl,
        eps_low=eps_low,
        eps_high=eps_high,
    )

    return stats

In [63]:
# ------------------------------------------------------------
# 9) DAPO settings
# ------------------------------------------------------------

num_dapo_iterations = 2

# NOTE Very imortant FOR DAPO  eps_high may be larger than eps_low
eps_low = 0.2
eps_high = 0.4

beta_kl = 0.04
lr = 1e-5

optimizer = torch.optim.Adam(
    policy_new.parameters(),
    lr=lr,
)


# ------------------------------------------------------------
# 10) Two prompt groups, batch size = 1 prompt group
# ------------------------------------------------------------

examples = [
    {
        "prompt": "cat is on the",
        "completions": [
            " wall and its sleeping.",
            " mat and it runs.",
            " floor.",
        ],
    },
    {
        "prompt": "2 + 2 =",
        "completions": [
            " 4.",
            " 5.",
            " four.",
        ],
    },
]


# ------------------------------------------------------------
# 11) Main DAPO flow
# ------------------------------------------------------------

for example_idx, ex in enumerate(examples, start=1):

    print("\n" + "=" * 80)
    print(f"DAPO EXAMPLE {example_idx} / BATCH SIZE = 1 PROMPT GROUP")
    print("=" * 80)

    prompt = ex["prompt"]
    completions = ex["completions"]

    print("Prompt:", repr(prompt))
    print("Completions:")
    for completion in completions:
        print("   ", repr(completion))

    # --------------------------------------------------------
    # A) Create rollout snapshot q for this prompt group
    # --------------------------------------------------------
    policy_old = copy.deepcopy(policy_new).to(device).eval()
    for p in policy_old.parameters():
        p.requires_grad_(False)

    print("\n[1] Rollout snapshot")
    print("q = policy_old = frozen copy of current policy_new")
    print("q is fixed for this prompt group.")

    # --------------------------------------------------------
    # B) Compute rewards for the completion group
    # --------------------------------------------------------
    print("\n[2] Compute group rewards")

    rewards = np.array(
        [toy_reward(prompt, completion) for completion in completions],
        dtype=np.float64,
    )

    print("    rewards:", rewards)

    # --------------------------------------------------------
    # C) Dynamic sampling gate
    # --------------------------------------------------------
    print("\n[3] Dynamic sampling gate")

    if not dynamic_sampling_ok(rewards, "binary"):
        print("Group rejected: all completions are correct or all are wrong.")
        print("In real DAPO, you would resample completions for this prompt.")
        continue

    print("    Group accepted: mixed correct/incorrect completions.")

    # --------------------------------------------------------
    # D) Collect rollout log-probs and group-relative advantages
    # --------------------------------------------------------
    print("\n[4] Collect rollout log-probs and group-relative advantages")

    rollout_items, group_stats = collect_dapo_rollout_group(
        policy_old,
        policy_reference,
        tokenizer,
        prompt,
        completions,
        rewards,
    )

    print("    mean_reward:", group_stats["mean_reward"])
    print("    std_reward: ", group_stats["std_reward"])
    print("    advantages: ", group_stats["advantages"])

    for item in rollout_items:
        print(
            "    completion:",
            repr(item["completion"]),
            "reward:",
            item["reward"],
            "advantage:",
            item["advantage"],
        )

    # --------------------------------------------------------
    # E) Before first DAPO update
    # --------------------------------------------------------
    print("\n[5] Before DAPO update")

    before = dapo_eval_stats(
        policy_new,
        tokenizer,
        prompt,
        rollout_items,
        beta_kl=beta_kl,
        eps_low=eps_low,
        eps_high=eps_high,
    )

    print("    rho mean/min/max:",
          before["mean_rho"], before["min_rho"], before["max_rho"])
    print("    objective:", before["objective"])
    print("    loss:     ", before["loss"])
    print("    mean_kl:  ", before["mean_kl"])
    print("    clipfrac: ", before["clipfrac"])

    # --------------------------------------------------------
    # F) DAPO update iterations on same rollout group
    # --------------------------------------------------------
    print("\n[6] DAPO update loop")
    print("    num_dapo_iterations:", num_dapo_iterations)
    print("    logp_old/q stays fixed during these iterations.")
    print("    logp_new/p is recomputed each iteration.")
    print("    eps_low:", eps_low, "eps_high:", eps_high)

    for dapo_iter in range(1, num_dapo_iterations + 1):

        stats = dapo_update_step(
            policy_new,
            optimizer,
            tokenizer,
            prompt,
            rollout_items,
            beta_kl=beta_kl,
            eps_low=eps_low,
            eps_high=eps_high,
        )

        print(f"\n    DAPO iteration {dapo_iter}")
        print("        rho mean/min/max:",
              stats["mean_rho"], stats["min_rho"], stats["max_rho"])
        print("        objective:", stats["objective"])
        print("        loss:     ", stats["loss"])
        print("        mean_kl:  ", stats["mean_kl"])
        print("        clipfrac: ", stats["clipfrac"])

    # --------------------------------------------------------
    # G) After update, evaluate same rollout group
    # --------------------------------------------------------
    print("\n[7] After DAPO update on same rollout group")

    after = dapo_eval_stats(
        policy_new,
        tokenizer,
        prompt,
        rollout_items,
        beta_kl=beta_kl,
        eps_low=eps_low,
        eps_high=eps_high,
    )

    print("    rho mean/min/max:",
          after["mean_rho"], after["min_rho"], after["max_rho"])
    print("    objective:", after["objective"])
    print("    loss:     ", after["loss"])
    print("    mean_kl:  ", after["mean_kl"])
    print("    clipfrac: ", after["clipfrac"])

    print("\n[8] Discard this rollout group")
    print("    Next prompt group will create a fresh policy_old snapshot.")


DAPO EXAMPLE 1 / BATCH SIZE = 1 PROMPT GROUP
Prompt: 'cat is on the'
Completions:
    ' wall and its sleeping.'
    ' mat and it runs.'
    ' floor.'

[1] Rollout snapshot
q = policy_old = frozen copy of current policy_new
q is fixed for this prompt group.

[2] Compute group rewards
    rewards: [1. 0. 1.]

[3] Dynamic sampling gate
    Group accepted: mixed correct/incorrect completions.

[4] Collect rollout log-probs and group-relative advantages


NameError: name 'encode_prompt_response' is not defined